# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AgrimJain-quantum/internship-remote/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

### Rule in plain words

Prioritize pages that either have meaningful search visibility but unusually low CTR for their position, or have gone 91+ days without an update. Rank triggered pages by 90-day impressions so higher-impact pages appear first.

### Two signal checks

**Signal 1 — CTR vs position:** `impressions_90d >= 300`, `0 < avg_position <= 20`, and `ctr < 0.5%`. Verdict: **CONFIRMED** when the signal-present bucket has the higher observed decline rate.

**Signal 2 — staleness:** `days_since_last_update >= 91`. Verdict: **CONFIRMED** when the 91+ day bucket has the higher observed decline rate.

The target is used only to audit these signals and evaluate the frozen baseline; it is never an input to the rule.

### Reason codes

- `ctr_fix`: visible page with low CTR.
- `stale`: 91+ days since last update.
- `ctr_fix_and_stale`: both signals are present.
- `no_action`: neither signal is present.

In [14]:
!git clone https://github.com/AgrimJain-quantum/internship-remote.git

Cloning into 'internship-remote'...
remote: Enumerating objects: 134, done.
remote: Counting objects: 100% (134/134), done.
remote: Compressing objects: 100% (91/91), done.
remote: Total 134 (delta 44), reused 92 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (134/134), 1.86 MiB | 2.32 MiB/s, done.
Resolving deltas: 100% (44/44), done.


In [15]:
%cd internship-remote

/content/internship-remote/work/notebooks/internship-remote


In [16]:
from pathlib import Path

print("Current directory:", Path.cwd())

data_path = Path("data/raw/content_refresh_anonymized.csv")

print("Dataset exists:", data_path.exists())
print("Dataset path:", data_path.resolve())

Current directory: /content/internship-remote/work/notebooks/internship-remote
Dataset exists: True
Dataset path: /content/internship-remote/work/notebooks/internship-remote/data/raw/content_refresh_anonymized.csv


In [17]:
%cd /content/internship-remote

/content/internship-remote


In [18]:
from pathlib import Path

print(Path.cwd())
print(Path("data/raw/content_refresh_anonymized.csv").exists())

/content/internship-remote
True


In [19]:
%run work/notebooks/w04_baseline_score.ipynb

/usr/local/lib/python3.12/dist-packages/nbformat/__init__.py:96: MissingIDFieldWarning: Cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  validate(nb)


In [20]:
import pandas as pd
import numpy as np

from pathlib import Path

_candidates = [
    Path("data/raw/content_refresh_anonymized.csv")
]
DATA_PATH = next((p for p in _candidates if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Could not locate data/raw/content_refresh_anonymized.csv")
df = pd.read_csv(DATA_PATH)

# Audit/evaluation target only. It is never used by the score.
if "is_declining_label" in df.columns:
    audit_label = df["is_declining_label"].astype(int)
elif "trend_direction" in df.columns:
    audit_label = (df["trend_direction"] == "down").astype(int)
else:
    raise ValueError("Starter CSV must contain trend_direction or is_declining_label.")

ctr_fix = (
    (df["impressions_90d"] >= 300)
    & (df["avg_position"] > 0)
    & (df["avg_position"] <= 20)
    & (df["ctr"] < 0.5)
)
stale = df["days_since_last_update"] >= 91

ctr_audit = (
    pd.DataFrame({"bucket": np.where(ctr_fix, "signal_present", "signal_absent"),
                  "label": audit_label})
    .groupby("bucket")
    .agg(n=("label", "size"), declining_rate=("label", "mean"))
    .reset_index()
)

stale_audit = (
    pd.DataFrame({"bucket": np.where(stale, "91+ days", "0-90 days"),
                  "label": audit_label})
    .groupby("bucket")
    .agg(n=("label", "size"), declining_rate=("label", "mean"))
    .reset_index()
)

display(ctr_audit.style.format({"declining_rate": "{:.2%}"}))
ctr_yes = ctr_audit.loc[ctr_audit["bucket"]=="signal_present","declining_rate"].iloc[0]
ctr_no = ctr_audit.loc[ctr_audit["bucket"]=="signal_absent","declining_rate"].iloc[0]
print("CTR-vs-position verdict:", "CONFIRMED" if ctr_yes > ctr_no else "OPPOSITE" if ctr_yes < ctr_no else "MIXED")

display(stale_audit.style.format({"declining_rate": "{:.2%}"}))
stale_yes = stale_audit.loc[stale_audit["bucket"]=="91+ days","declining_rate"].iloc[0]
stale_no = stale_audit.loc[stale_audit["bucket"]=="0-90 days","declining_rate"].iloc[0]
print("Staleness verdict:", "CONFIRMED" if stale_yes > stale_no else "OPPOSITE" if stale_yes < stale_no else "MIXED")

,bucket,n,declining_rate
0,signal_absent,19270,49.01%
1,signal_present,10730,63.54%


CTR-vs-position verdict: CONFIRMED


,bucket,n,declining_rate
0,0-90 days,20655,51.20%
1,91+ days,9345,60.85%


Staleness verdict: CONFIRMED


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [21]:
from pathlib import Path

out_candidates = [Path("work/outputs"), Path("../../work/outputs")]
out_dir = next((p for p in out_candidates if p.exists() or p.parent.exists()), Path("work/outputs"))
out_dir.mkdir(parents=True, exist_ok=True)
out_dir.mkdir(parents=True, exist_ok=True)

# Transparent, unfitted score: trigger × 90-day impressions.
df["ctr_fix_signal"] = ctr_fix.astype(int)
df["stale_signal"] = stale.astype(int)
df["score"] = ((ctr_fix | stale).astype(int) * df["impressions_90d"])

def make_reason(row):
    if row["ctr_fix_signal"] and row["stale_signal"]:
        return "ctr_fix_and_stale"
    if row["ctr_fix_signal"]:
        return "ctr_fix"
    if row["stale_signal"]:
        return "stale"
    return "no_action"

def make_action(row):
    if row["ctr_fix_signal"] and row["stale_signal"]:
        return "review_ctr_and_refresh"
    if row["ctr_fix_signal"]:
        return "review_ctr"
    if row["stale_signal"]:
        return "refresh"
    return "no_action"

df["reason_code"] = df.apply(make_reason, axis=1)
df["action"] = df.apply(make_action, axis=1)
df["_audit_label"] = audit_label

ranked = df.sort_values(
    ["score", "impressions_90d", "content_id"],
    ascending=[False, False, True]
).reset_index(drop=True)
ranked["rank"] = np.arange(1, len(ranked)+1)

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

print(f"Base rate: {ranked['_audit_label'].mean():.2%}")
for k in [10, 20, 50, 100]:
    print(f"Precision@{k}: {precision_at_k(ranked['score'], ranked['_audit_label'], k):.2%}")

queue_cols = [
    "rank", "content_id", "score", "reason_code", "action",
    "impressions_90d", "ctr", "avg_position", "days_since_last_update"
]
ranked[queue_cols].to_csv(out_dir/"baseline_action_score.csv", index=False)
print(f"Wrote {len(ranked):,} rows to work/outputs/baseline_action_score.csv")
display(ranked[queue_cols].head(10))


Base rate: 54.21%
Precision@10: 70.00%
Precision@20: 55.00%
Precision@50: 44.00%
Precision@100: 42.00%
Wrote 30,000 rows to work/outputs/baseline_action_score.csv


,rank,content_id,score,reason_code,action,impressions_90d,ctr,avg_position,days_since_last_update
0,1,content_5fe46e04994d,517715,ctr_fix_and_stale,review_ctr_and_refresh,517715,0.14,4.2,104
1,2,content_aaef01a50def,517109,ctr_fix,review_ctr,517109,0.25,5.4,22
2,3,content_8c19996aa890,509252,ctr_fix,review_ctr,509252,0.15,2.5,20
3,4,content_4c36c775b818,463103,ctr_fix,review_ctr,463103,0.41,2.3,20
4,5,content_2dba2b1f9536,443434,stale,refresh,443434,0.21,27.9,104
5,6,content_1a9e894be2e2,416180,ctr_fix,review_ctr,416180,0.23,4.0,22
6,7,content_2c2606c5d176,347399,stale,refresh,347399,0.53,4.2,104
7,8,content_db5989a78dd3,345111,ctr_fix,review_ctr,345111,0.21,5.4,20
8,9,content_cb112fce36be,309910,ctr_fix_and_stale,review_ctr_and_refresh,309910,0.16,5.6,104
9,10,content_9532f197bbc8,309192,stale,refresh,309192,0.87,2.0,104


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

The review is deliberately skeptical: a high score is not treated as proof that the recommended action is correct.

In [22]:
def review_line(row):
    explanations = {
        "ctr_fix_and_stale": "low CTR for a visible position and 91+ days since update",
        "ctr_fix": "low CTR for a visible position with meaningful impressions",
        "stale": "91+ days since the last update",
        "no_action": "no baseline trigger"
    }
    wrong_if = {
        "ctr_fix_and_stale": "the low CTR is explained by query mix or intent, or the page is intentionally evergreen",
        "ctr_fix": "CTR is appropriate for the query mix, or the apparent position is too unstable to support the recommendation",
        "stale": "the page is intentionally evergreen, or a recent update was not recorded in the source data",
        "no_action": "a meaningful issue exists that these two signals do not capture"
    }
    return pd.Series({
        "rank": int(row["rank"]),
        "action": row["action"],
        "reason_code": row["reason_code"],
        "why": explanations[row["reason_code"]],
        "confidence_note": "higher" if row["rank"] <= 10 else "moderate",
        "what_would_make_it_wrong": wrong_if[row["reason_code"]]
    })

top20_review = ranked.head(20).apply(review_line, axis=1)
display(top20_review)
print("The first 10 rows satisfy the assignment's top-10 review requirement; the supplied skeleton asks for top 20, so all 20 are reviewed.")

,rank,action,reason_code,why,confidence_note,what_would_make_it_wrong
0,1,review_ctr_and_refresh,ctr_fix_and_stale,low CTR for a visible position and 91+ days si...,higher,the low CTR is explained by query mix or inten...
1,2,review_ctr,ctr_fix,low CTR for a visible position with meaningful...,higher,"CTR is appropriate for the query mix, or the a..."
2,3,review_ctr,ctr_fix,low CTR for a visible position with meaningful...,higher,"CTR is appropriate for the query mix, or the a..."
3,4,review_ctr,ctr_fix,low CTR for a visible position with meaningful...,higher,"CTR is appropriate for the query mix, or the a..."
4,5,refresh,stale,91+ days since the last update,higher,"the page is intentionally evergreen, or a rece..."
5,6,review_ctr,ctr_fix,low CTR for a visible position with meaningful...,higher,"CTR is appropriate for the query mix, or the a..."
6,7,refresh,stale,91+ days since the last update,higher,"the page is intentionally evergreen, or a rece..."
7,8,review_ctr,ctr_fix,low CTR for a visible position with meaningful...,higher,"CTR is appropriate for the query mix, or the a..."
8,9,review_ctr_and_refresh,ctr_fix_and_stale,low CTR for a visible position and 91+ days si...,higher,the low CTR is explained by query mix or inten...
9,10,refresh,stale,91+ days since the last update,higher,"the page is intentionally evergreen, or a rece..."


The first 10 rows satisfy the assignment's top-10 review requirement; the supplied skeleton asks for top 20, so all 20 are reviewed.


## 4. Weak picks + leakage check

The baseline is intentionally simple, so weak picks are expected. A high score can still be wrong when the page is evergreen, CTR is explained by query intent, or the position signal is unstable.

The score uses only current snapshot fields: `impressions_90d`, `ctr`, `avg_position`, and `days_since_last_update`. The audit target is never included in the score.

### Programmatic leakage checks

In [23]:
score_inputs = {
    "impressions_90d", "ctr", "avg_position", "days_since_last_update"
}
forbidden = {
    "trend_direction", "trend_pct", "is_declining_label",
    "content_id", "client_id"
}

print("Score inputs:", sorted(score_inputs))
print("Forbidden fields excluded from score:",
      sorted(forbidden.intersection(df.columns)))

assert not score_inputs.intersection(forbidden)
assert "trend_direction" not in score_inputs
assert "trend_pct" not in score_inputs
assert "is_declining_label" not in score_inputs
assert "content_id" not in score_inputs
assert "client_id" not in score_inputs

print("Leakage check: PASS")
print("Weak-pick reminder: inspect low-volume or intent-sensitive cases manually; this rule is decision-support, not proof of causation.")

Score inputs: ['avg_position', 'ctr', 'days_since_last_update', 'impressions_90d']
Forbidden fields excluded from score: ['client_id', 'content_id', 'trend_direction', 'trend_pct']
Leakage check: PASS
Weak-pick reminder: inspect low-volume or intent-sensitive cases manually; this rule is decision-support, not proof of causation.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.